# NexPlay — riesgo de arrepentimiento temprano al comprar un videojuego

Diplomado en Ciencia de Datos, FES Acatlán (UNAM) — Módulo V.

Este notebook corre de principio a fin en un Colab limpio: descarga un extracto de datos publicado como asset de un GitHub Release (con verificación SHA-256) y clona el código de entrenamiento del repo a un tag fijo. No usa Google Drive, no pide credenciales y no ejecuta la ingesta de Steam — todo eso ya ocurrió para producir el extracto.

**Narrativa:** Problema → Datos → EDA → Calidad de datos → Ingeniería de variables → Modelo → Experimento de privacidad → Conclusiones.

## 1. Problema

Cuando alguien compra un videojuego en Steam, tiene una ventana de 120 minutos de juego para pedir reembolso. NexPlay busca estimar, **antes de la compra**, el riesgo de que un jugador se arrepienta tempranamente — para eso, antes de que exista una compra real, solo puede usar dos tipos de información: lo que ya se sabe del juego (precio, descuento, recepción de la crítica) y lo que el jugador declara de sí mismo en un formulario de alta.

No observamos arrepentimiento directamente — Steam no pregunta "¿te arrepentiste?". Usamos una señal *proxy*: reseñas donde el autor jugó poco y calificó negativo.

$$Y = 1 \iff \texttt{playtime\_at\_review} < 120 \text{ min} \ \wedge\ \texttt{voted\_up} = 0$$

El umbral de 120 minutos no es arbitrario: es exactamente la ventana de reembolso de Steam. A esta señal la llamamos **arrepentimiento temprano**, nunca "abandono" ni "insatisfacción" — son cosas distintas que esta variable no puede distinguir.

## 2. Datos

### 2.1 Origen

119k+ reseñas ingeridas desde la API pública `appreviews` de Steam sobre un catálogo curado de juegos (`appids.txt` en el repo), pensado en capas:

- **Capa A** — contraste de dificultad/experiencia: juegos que la comunidad adora pero que son duros para alguien nuevo (Dark Souls, Kenshi, Dwarf Fortress) contra puertas de entrada (Stardew Valley, Hades, Portal). Sin este contraste la variable objetivo podría no tener varianza.
- **Capa B** — intersección con el corpus de Metacritic, para comparar motivos entre plataformas (fuera del alcance de este notebook).
- **Capa C** — brecha entre expectativa y recepción: lanzamientos AAA con recibimiento muy disparejo.

### 2.2 Extracto reproducible

`nexplay.db` (SQLite) no se publica: tiene texto de reseñas y vive en `datos/`, fuera de git. En su lugar, `extracto_datos.py` (en el repo) genera un extracto mínimo en Parquet — sin texto, sin `steamid`, sin nada que no haga falta para esta narrativa — y lo publicamos como *asset* de un GitHub Release con **tag fijo** (nunca `latest`, para que esta celda siga funcionando igual dentro de un año). Este notebook descarga ese asset y valida su SHA-256 antes de tocarlo.

El extracto trae: `appid`, `nombre` (para nombrar juegos concretos en el EDA), `playtime_at_review`, `voted_up`, `timestamp_created` — reconstruyen el target y agrupan el GroupKFold — y `num_games_owned`, `es_gratis`, `precio_final`, `descuento`, `metacritic`: las columnas crudas detrás de las seis variables del modelo de producción, más `num_games_owned` cruda (no es feature del modelo, pero sin ella no se puede reproducir la bandera de privacidad de perfil ni el experimento de la sección 7).

In [ ]:
# Colab ya trae pandas, numpy, scikit-learn y pyarrow con binarios precompilados que
# coinciden entre si. Instalar versiones fijas con pip (numpy==..., scipy==..., etc.)
# rompe esa compatibilidad binaria y produce errores como
# "ImportError: cannot import name '_slice' from 'numpy._core.umath'".
# Por eso este notebook no fija ni reinstala numpy/scipy/scikit-learn: si de verdad
# falta algun paquete (no deberia, en un Colab estandar), se instala solo, sin tocarlos.
import importlib.util
import subprocess
import sys

requeridos = {"pyarrow": "pyarrow", "requests": "requests"}
faltantes = [paquete for paquete, modulo in requeridos.items() if importlib.util.find_spec(modulo) is None]
if faltantes:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltantes], check=True)

In [ ]:
# Constancia del entorno en el que corrio este notebook (Colab ya trae estos paquetes
# precompilados; no se instala nada aqui, solo se imprime lo que Colab ya tiene).
import sys

import numpy
import pandas
import pyarrow
import requests
import sklearn

print(f"python       {sys.version.split()[0]}")
print(f"numpy        {numpy.__version__}")
print(f"pandas       {pandas.__version__}")
print(f"scikit-learn {sklearn.__version__}")
print(f"pyarrow      {pyarrow.__version__}")
print(f"requests     {requests.__version__}")

### 2.3 Código compartido, no duplicado

El entrenamiento (`construir_features`, `construir_pipeline`, `evaluar_gkf`, `comparar_variantes_privacidad`) vive en `entrenar_baseline.py`, en el repo. Es un script normal — su `main()` está protegido por `if __name__ == "__main__":`, así que importarlo no ejecuta nada por sí solo. Este notebook clona el repo a un **tag/commit fijo** (no la rama por defecto, que puede cambiar) e importa esas mismas funciones: nunca copia la lógica.

In [ ]:
# --- Configuracion fija: repo publico y tag del release (NO "latest", NO una rama) ---
GITHUB_REPO = "fernandoaxelramirezgomez-coder/nexplay"
GITHUB_REF = "data-v2"

PARQUET_URL = f"https://github.com/{GITHUB_REPO}/releases/download/{GITHUB_REF}/nexplay_extracto.parquet"
# sha256 real del extracto publicado en ese release. Si se regenera el extracto y se sube un
# nuevo asset (idealmente con un tag nuevo), este valor tiene que actualizarse junto con el.
PARQUET_SHA256 = "025323e0d0e99031ed940e0a12218ed8e5dd9068bd4b0aedb2a805b01be2a0d1"

In [ ]:
import os

if os.path.isdir("repo_nexplay"):
    print("repo_nexplay ya existe, no se vuelve a clonar.")
else:
    !git clone --quiet --branch {GITHUB_REF} --depth 1 https://github.com/{GITHUB_REPO}.git repo_nexplay

In [ ]:
import sys

sys.path.insert(0, "repo_nexplay")

from entrenar_baseline import (  # noqa: E402 (import tras sys.path.insert, a proposito)
    N_SPLITS,
    SEMILLA,
    comparar_variantes_privacidad,
    construir_features,
    construir_pipeline,
    evaluar_gkf,
)

In [ ]:
import hashlib
from pathlib import Path

import requests

DATA_PATH = Path("nexplay_extracto.parquet")

resp = requests.get(PARQUET_URL, timeout=60)
resp.raise_for_status()
DATA_PATH.write_bytes(resp.content)

sha256_obtenido = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
if sha256_obtenido != PARQUET_SHA256:
    raise ValueError(
        f"SHA-256 no coincide: esperado {PARQUET_SHA256}, obtenido {sha256_obtenido}. "
        "El asset del release pudo cambiar o la descarga se corrompio; no seguir sin verificarlo."
    )
print(f"descarga verificada: {DATA_PATH.stat().st_size / 1024:.1f} KB, sha256 OK")

In [ ]:
import pandas as pd

df = pd.read_parquet(DATA_PATH)
df["y"] = ((df["playtime_at_review"] < 120) & (df["voted_up"] == 0)).astype(int)

print(f"filas={len(df)}  juegos={df['appid'].nunique()}")
df.head()

## 3. EDA

### 3.1 Prevalencia global

In [ ]:
prevalencia_global = df["y"].mean()
print(f"prevalencia global de arrepentimiento temprano: {prevalencia_global:.4f} ({100*prevalencia_global:.2f}%)")

Clase muy desbalanceada — por eso la métrica de validación es PR-AUC, no accuracy (un modelo que siempre dice "no" acierta más del 97% de las veces sin decir nada útil).

### 3.2 Prevalencia por juego

La Capa A se armó con una hipótesis concreta: juegos duros para un jugador nuevo (Dark Souls, Kenshi, Dwarf Fortress) deberían mostrar más arrepentimiento temprano que puertas de entrada (Stardew Valley, Hades, Portal).

In [ ]:
por_juego = (
    df.groupby("nombre")["y"]
    .agg(n="size", prevalencia="mean")
    .sort_values("prevalencia")
)

duros = ["DARK SOULS™: REMASTERED", "Kenshi", "Dwarf Fortress"]
accesibles = ["Portal", "Stardew Valley", "Hades"]
por_juego.loc[duros + accesibles]

Los tres juegos elegidos por "difíciles para un novato" tienen prevalencia tan baja como los de entrada — entre 0.3% y 1.3%, todos por debajo de la media global. La dificultad del juego para alguien nuevo, sola, no separa nada.

In [ ]:
print("--- menor prevalencia ---")
display(por_juego.head(8))
print("--- mayor prevalencia ---")
display(por_juego.tail(8))

Lo que sí separa con fuerza son lanzamientos con recepción muy pareja/floja frente a la expectativa (*The Lord of the Rings: Gollum*, *WILD HEARTS*, *Redfall*, *Suicide Squad: Kill the Justice League*, *Skull and Bones*, *Battlefield 2042*), con prevalencias entre 10% y 27% — 5 a 12 veces la media global, y muy por encima de cualquier juego "difícil" de la Capa A. Es exactamente la intuición detrás de la Capa C del catálogo (brecha expectativa vs. recepción), no de la Capa A.

**Encontramos que** el eje que separa a los juegos no es qué tan duro es para un jugador nuevo, sino qué tan bien fue recibido en su lanzamiento — algo que se puede leer, en buena medida, del lado del juego (precio, descuento, nota de Metacritic) sin necesitar casi nada del perfil de quien compra. Esto reencuadra el proyecto: en vez de perfilar exhaustivamente al jugador, el modelo debe apoyarse primero en lo que transfiere del juego, y tratar el perfil declarado del jugador como un ajuste secundario — no como el eje principal. Lo confirmamos con números en la sección 6 (Modelo): el conjunto sin ninguna variable de jugador retiene casi todo el PR-AUC del conjunto completo de producción.

## 4. Calidad de datos

In [ ]:
priv_pct = (df["num_games_owned"] == 0).mean()
metacritic_na_pct = df["metacritic"].isna().mean()
precio_na_pct = df["precio_final"].isna().mean()

print(f"perfiles con num_games_owned == 0: {100*priv_pct:.1f}%")
print(f"juegos sin nota de metacritic: {100*metacritic_na_pct:.1f}%")
print(f"filas sin precio_final: {100*precio_na_pct:.1f}%")

**`num_games_owned == 0` es bandera de privacidad, no biblioteca vacía.** Steam no distingue "no tiene juegos" de "su biblioteca es privada" — ambos casos llegan como 0. Con ~60% de las reseñas en ese caso, imputarlo como "cero juegos" sería tratar como información algo que es, en su mayoría, ausencia de información. Por eso el proyecto lo trata con un flag explícito en vez de imputarlo en silencio (ver sección 7 — y por qué ese flag, ya evaluado, no quedó en el modelo de producción).

In [ ]:
# Dos juegos pagos sin precio_final (no son gratis, pero el campo llego nulo en la ingesta):
mask_precio_raro = df["precio_final"].isna() & (df["es_gratis"] == 0)
df.loc[mask_precio_raro, "nombre"].unique()

Son 2 de 83 juegos (3,000 de 123,972 filas, 2.4%) — probablemente removidos o repriceados en Steam entre la ingesta y hoy. `construir_features` los trata con `fillna(0)`, igual que a los juegos gratis, y ahí está el problema: para un juego gratuito el 0 es correcto, para uno de pago no. Es una limitación conocida; se cuantifica al final de esta sección.

**Nada de fuga temporal.** El extracto no trae `playtime_forever` ni ningún campo que solo exista porque el autor ya reseñó (`num_reviews`, `steam_purchase`, etc.) — esas columnas solo se usan en el conjunto `'completo'` de `entrenar_baseline.py`, que es apenas un chequeo interno de "¿hay señal?" y nunca llega a producción.

In [ ]:
fechas = pd.to_datetime(df["timestamp_created"], unit="s")
print(f"reseñas entre {fechas.min().date()} y {fechas.max().date()}")

assert df["playtime_at_review"].ge(0).all(), "playtime_at_review negativo"
assert df["voted_up"].isin([0, 1]).all(), "voted_up fuera de {0,1}"
assert df["descuento"].dropna().between(0, 100).all(), "descuento fuera de [0,100]"
print("chequeos de rango: OK")

### Limitación conocida: dos juegos de pago con el precio imputado como 0

En 2 de los 83 juegos —**Grand Theft Auto V Legacy** y **New World: Aeternum**— `appdetails` no devolvió `price_overview`, así que `precio_final` llegó nulo aunque **no son gratuitos** (`es_gratis = 0`). `construir_features` llena ese nulo con 0 y `api/scoring.py` hace lo mismo al puntuar: el modelo se entrenó con el 0 y lo sigue viendo en inferencia. Afecta a **3,000 de 123,972 filas** (2.4%; 1,500 por juego, lo mismo que pesa un juego típico).

Para un juego gratuito el 0 es correcto; para uno de pago no. "De pago y a precio 0" no existe entre los juegos con precio real (el mínimo es 59 MXN): queda a 4.2 desviaciones por debajo de la media, el modelo lo lee como "muy barato" y le resta ≈1.07 al log-odds de esos dos juegos sin que sea información del juego.

**Prueba de sensibilidad** (la celda de abajo la reproduce; usa el modelo de la sección 6):

- **Imputar sin reentrenar** (mediana de precio de los juegos de pago, 400 MXN, solo al puntuar): **ninguna banda cambia**. GTA V Legacy pasa de 0.078 a 0.206 y New World: Aeternum de 0.448 a 0.713, y cada uno se queda en su banda (bajo y alto). Igual con el precio mínimo o el máximo observados (59 a 1,599 MXN). Cambia la posición dentro de la banda, no la banda.
- **Reentrenar con la mediana imputada:** el PR-AUC no mejora (0.0709 → 0.0670, media entre folds; la desviación entre folds es ~0.027, así que la diferencia es ruido) y **10 juegos cambian de banda** —ninguno de los dos afectados, todos cerca de un corte de banda, donde hay muchos juegos con scores casi iguales—. Además los coeficientes de precio y de gratuidad se mueven mucho (`log_precio_final` +0.26 → +0.61; `es_gratis` +0.09 → +0.49): esas 3,000 filas pesan en el modelo más de lo que sugiere su 2.4%, porque están muy lejos de las demás en precio.

**Decisión:** el modelo se queda como está, con el 0 igual en entrenamiento e inferencia, y esto se documenta como limitación en vez de reentrenar: reentrenar no mejora la métrica y reacomoda juegos que no tienen el problema. Lo que sí se corrigió es lo que se le dice a quien usa la UI: la ficha de esos dos juegos ya no muestra el precio como factor del riesgo, porque "precio por debajo del promedio" no describe al juego.

In [ ]:
import numpy as np

from entrenar_modelo import _scores_oof  # noqa: E402 (mismos cortes por terciles que usa la API)

sin_precio = df["precio_final"].isna() & (df["es_gratis"] == 0)
precios = df.loc[(df["es_gratis"] == 0) & df["precio_final"].notna()].drop_duplicates("appid")["precio_final"]
mediana_precio = precios.median()
nombres = df.drop_duplicates("appid").set_index("appid")["nombre"]

print(f"juegos de pago sin precio: {list(df.loc[sin_precio, 'nombre'].unique())}")
print(f"filas afectadas: {sin_precio.sum()} de {len(df)} ({100 * sin_precio.mean():.1f}%)")
print(f"precio_final en centavos — minimo={precios.min():.0f}  mediana={mediana_precio:.0f}  maximo={precios.max():.0f}\n")


def catalogo_neutro(X, grupos):
    """Una fila por juego con el perfil neutro de api/catalogo.py (5 compras al año)."""
    juegos = X.groupby(grupos).first()
    juegos["log_num_games_owned"] = np.log1p(5)
    return juegos


def scores_y_bandas(modelo, juegos, cortes):
    scores = pd.Series(modelo.predict_proba(juegos)[:, 1], index=juegos.index)
    bandas = np.select([scores < cortes[0], scores < cortes[1]], ["bajo", "medio"], "alto")
    return scores, pd.Series(bandas, index=juegos.index)


# Hoy: el faltante entra como 0, en entrenamiento y en inferencia
X_hoy, y_hoy, g_hoy = construir_features(df, conjunto="compra")
cortes_hoy = np.percentile(_scores_oof(X_hoy, y_hoy, g_hoy), [100 / 3, 200 / 3])
modelo_hoy = construir_pipeline().fit(X_hoy, y_hoy)
juegos_hoy = catalogo_neutro(X_hoy, g_hoy)
scores_hoy, bandas_hoy = scores_y_bandas(modelo_hoy, juegos_hoy, cortes_hoy)
afectados = juegos_hoy.index.isin(df.loc[sin_precio, "appid"])

escalador, clf = modelo_hoy.named_steps["escalar"], modelo_hoy.named_steps["clf"]
i_precio = list(X_hoy.columns).index("log_precio_final")
z_cero = (0 - escalador.mean_[i_precio]) / escalador.scale_[i_precio]
print(f"un precio 0 queda a {z_cero:.1f} desviaciones de la media; aporte de log_precio_final al log-odds: {clf.coef_[0][i_precio] * z_cero:+.2f}\n")

# (a) Sin reentrenar: el modelo actual, con un precio imputado solo al puntuar
print("(a) sin reentrenar, precio imputado al puntuar:")
for etiqueta, precio in (("minimo", precios.min()), ("mediana", mediana_precio), ("maximo", precios.max())):
    juegos_a = juegos_hoy.copy()
    juegos_a.loc[afectados, "log_precio_final"] = np.log1p(precio)
    scores_a, bandas_a = scores_y_bandas(modelo_hoy, juegos_a, cortes_hoy)
    print(f"  precio={etiqueta:<8} juegos que cambian de banda: {(bandas_a != bandas_hoy).sum()}")
    if etiqueta == "mediana":
        for appid in juegos_hoy.index[afectados]:
            print(f"      {nombres[appid]}: score {scores_hoy[appid]:.4f} ({bandas_hoy[appid]}) -> {scores_a[appid]:.4f} ({bandas_a[appid]})")

# (b) Reentrenando con la mediana imputada en las filas afectadas
df_imp = df.copy()
df_imp.loc[sin_precio, "precio_final"] = mediana_precio
X_imp, y_imp, g_imp = construir_features(df_imp, conjunto="compra")
print("\n(b) reentrenando con la mediana imputada:")
pr_hoy = evaluar_gkf(construir_pipeline(), X_hoy, y_hoy, g_hoy, "hoy")
pr_imp = evaluar_gkf(construir_pipeline(), X_imp, y_imp, g_imp, "imputado")
cortes_imp = np.percentile(_scores_oof(X_imp, y_imp, g_imp), [100 / 3, 200 / 3])
modelo_imp = construir_pipeline().fit(X_imp, y_imp)
_, bandas_imp = scores_y_bandas(modelo_imp, catalogo_neutro(X_imp, g_imp), cortes_imp)
cambian = bandas_hoy.index[bandas_hoy != bandas_imp]

coef_hoy = pd.Series(modelo_hoy.named_steps["clf"].coef_[0], index=X_hoy.columns)
coef_imp = pd.Series(modelo_imp.named_steps["clf"].coef_[0], index=X_imp.columns)
for variable in ("log_precio_final", "es_gratis"):
    print(f"  coeficiente de {variable}: {coef_hoy[variable]:+.4f} -> {coef_imp[variable]:+.4f}")
print(f"  PR-AUC (media entre folds): hoy={pr_hoy.mean():.4f}  imputado={pr_imp.mean():.4f}  (std entre folds ~{pr_hoy.std():.3f})")
print(f"  juegos que cambian de banda: {len(cambian)} de {len(bandas_hoy)}")
for appid in cambian:
    print(f"      {nombres[appid]}: {bandas_hoy[appid]} -> {bandas_imp[appid]}")

## 5. Ingeniería de variables

`construir_features(df, conjunto="compra")` arma exactamente las seis variables del modelo de producción: solo lo que se conoce **antes** de que el jugador juegue — lo que el catálogo ya sabe del juego, más lo que el formulario de alta declara del jugador. Nada que dependa de que la reseña ya exista.

In [ ]:
X, y, grupos = construir_features(df, conjunto="compra")
print("features:", list(X.columns))
X.assign(y=y).sample(5, random_state=SEMILLA)

- `log_num_games_owned`: `log1p` sobre un conteo con cola larga (unos pocos perfiles declaran cientos de juegos). En producción, `compras_al_anio` del formulario sustituye a `num_games_owned` — es la variable que el jugador sí puede declarar sin depender de una cuenta externa.
- `es_gratis`, `descuento`: ya vienen acotadas (0/1 y 0-100), sin transformar.
- `log_precio_final`: mismo `log1p`, precios van de centavos a cientos de pesos.
- `metacritic_disponible` + `metacritic`: el 27% sin nota se imputa con la mediana, pero marcado con un flag — el modelo puede aprender que "no tiene nota" es distinto de "tiene una nota mediocre", en vez de mezclarlos en silencio. El flag marca **ausencia de cobertura crítica** (Metacritic no agregó reseñas para ese título), no una propiedad del juego — no debe leerse como señal de calidad.

`grupos` es `appid`: es la clave de todo el esquema de validación de la sección 6.

## 6. Modelo

Regresión logística con `class_weight="balanced"`, sin tuning — el piso que hay que superar, no el modelo final. Validación con `GroupKFold` por `appid`: cada fold deja afuera juegos completos, así el PR-AUC mide generalización a juegos que el modelo nunca vio, no memorización de un juego particular.

In [ ]:
from sklearn.dummy import DummyClassifier

print(f"GroupKFold con N_SPLITS={N_SPLITS}, semilla={SEMILLA}\n")

print("=== conjunto 'compra' (modelo de produccion) ===")
trivial = evaluar_gkf(DummyClassifier(strategy="prior"), X, y, grupos, "compra/trivial")
logreg = evaluar_gkf(construir_pipeline(), X, y, grupos, "compra/logreg")

In [ ]:
print("=== conjunto 'juego' (solo lado del juego, sin nada del jugador) ===")
X_juego, y_juego, grupos_juego = construir_features(df, conjunto="juego")
print("features:", list(X_juego.columns), "\n")

trivial_juego = evaluar_gkf(DummyClassifier(strategy="prior"), X_juego, y_juego, grupos_juego, "juego/trivial")
logreg_juego = evaluar_gkf(construir_pipeline(), X_juego, y_juego, grupos_juego, "juego/logreg")

In [ ]:
caida = 1 - logreg_juego.mean() / logreg.mean()
print(f"compra: PR-AUC={logreg.mean():.4f}  juego: PR-AUC={logreg_juego.mean():.4f}")
print(f"quitar TODO el lado del jugador cuesta solo {100*caida:.1f}% de PR-AUC")

Confirma lo que sugería la sección 3: sacar por completo el lado del jugador (`compras_al_anio` vía `log_num_games_owned`) apenas mueve el PR-AUC. La mayor parte de la señal viene del juego, no de quién compra — coherente con `nota_plataforma` en la API: "el lado del juego transfiere".

In [ ]:
pipeline_final = construir_pipeline()
pipeline_final.fit(X, y)
coefs = pd.Series(pipeline_final.named_steps["clf"].coef_[0], index=X.columns).sort_values()
coefs

`class_weight="balanced"` reescala las clases para que el modelo aprenda con la minoría, pero eso significa que `predict_proba` **ya no es una probabilidad calibrada** — es un score útil para *ordenar* riesgo relativo, no para leerse como "38% de probabilidad de arrepentimiento". Por eso la API (`api/scoring.py`) y la UI solo exponen un nivel (BAJO/MEDIO/ALTO, calibrado por tercios de la distribución de scores de validación) y nunca un porcentaje — mostrarlo como probabilidad induciría a error.

## 7. Experimento de privacidad

`privacidad_perfil` (`num_games_owned == 0`, sección 4) se evaluó como feature explícita del conjunto `'compra'` antes de que existiera el modelo de producción. `comparar_variantes_privacidad` reproduce esa comparación ya aprobada, sin volver a experimentar: tres variantes con `GroupKFold` sobre las mismas features de `'compra'` más/menos esa bandera.

In [ ]:
resultados_privacidad = comparar_variantes_privacidad(df)

In [ ]:
for variante in ("con_privacidad", "sin_privacidad", "solo_publico"):
    r = resultados_privacidad[variante]
    print(f"{variante:<15} media={r.mean():.4f}  std={r.std():.4f}")

diferencia = resultados_privacidad["con_privacidad"].mean() - resultados_privacidad["sin_privacidad"].mean()
desviacion = resultados_privacidad["con_privacidad"].std()
print(f"\ndiferencia con/sin bandera: {diferencia:.4f}")
print(f"desviacion entre folds: {desviacion:.4f}")
print(f"tamaño de 'solo_publico' (privacidad_perfil == 0): {resultados_privacidad['n_solo_publico']} filas")

La diferencia de PR-AUC entre incluir la bandera y no incluirla (~0.0036) es un orden de magnitud menor que la desviación entre folds (~0.027): no hay señal real, es ruido de muestreo. El subconjunto `solo_publico` (perfiles no privados, 40% de las filas) además muestra folds muy inestables (0.014 a 0.156) por tener menos positivos por fold — otra razón para no construir una regla especial alrededor de él.

**Decisión ya tomada y aplicada:** `privacidad_perfil` se sacó del conjunto `'compra'` en `entrenar_baseline.py` (se conserva solo en `'completo'`, donde se originó la comparación) y `modelo/nexplay.pkl` se re-entrenó sin ella — es el modelo que corre en `api/scoring.py` hoy.

## 8. Conclusiones

- **El target es una señal proxy, no arrepentimiento observado.** `playtime_at_review < 120` y `voted_up == 0` es lo más cercano que da la API de Steam a "esto no era lo que esperaba", pero no es lo mismo que preguntarle al jugador.
- **El reencuadre central de este proyecto:** el riesgo depende mucho más del juego (precio, descuento, recepción de crítica) que de quién lo compra. El conjunto `'juego'` —sin ninguna variable de jugador— retiene casi todo el PR-AUC del conjunto de producción `'compra'`: quitar por completo el lado del jugador aporta solo 2.1% de PR-AUC (sección 6), una diferencia del mismo orden que el ruido entre folds — no una señal robusta. En la práctica, **el modelo estima el riesgo del título**; el perfil declarado en el formulario sirve para **filtrar afinidad** (qué tanto encaja el juego con lo que el jugador dice que tolera y prefiere), no para modificar el score.
- **`privacidad_perfil` se descartó con evidencia, no por intuición**: la diferencia de PR-AUC al quitarla es un orden de magnitud menor que el ruido entre folds.
- **El modelo de producción (`modelo/nexplay.pkl`) es un piso, no un techo**: regresión logística sin tuning, seis variables, PR-AUC ~0.07 contra una prevalencia de 2.2% (~3-4x mejor que un clasificador trivial). Con `class_weight="balanced"` el score ordena riesgo relativo pero no es una probabilidad calibrada — por eso la API expone un nivel (BAJO/MEDIO/ALTO, por tercios de la distribución de validación) y no un porcentaje.
- **Límites conocidos:** todo el entrenamiento es de reseñas de Steam (PC); no existe una fuente propia de PlayStation/Xbox/Nintendo, así que el lado del juego transfiere a otras plataformas pero el modelo no fue validado ahí (`nota_plataforma` en la API lo advierte). Tampoco se usó texto de reseña ni el corpus de Metacritic (fuente secundaria, solo para comparar motivos, nunca para entrenar).
- **Próximo paso natural:** el conjunto `'completo'` (con `num_reviews`, etc.) muestra que hay algo más de señal cuando se conocen datos posteriores a la reseña — pero eso es fuga en producción. Vale la pena explorar features de texto o de comportamiento temprano dentro de la ventana de reembolso, no post-hoc.